In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from fontTools.diff import color
from matplotlib import figure
from datetime import datetime, timedelta

from scipy.cluster.hierarchy import centroid
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

In [ ]:
df = pd.read_csv("../data/main_data.csv")

# Clleaning data & formating it

In [ ]:
df = df.dropna(subset=["CustomerID"]).copy()

df["CustomerID"] = df["CustomerID"].astype(int)
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"]).dt.normalize()
df["UnitPrice"] = pd.to_numeric(
    df["UnitPrice"].astype(str).str.replace(",", ".", regex=False),
    errors="raise",
)

df = df[~df["InvoiceNo"].astype(str).str.startswith("C")]
df = df[(df["Quantity"] > 0) & (df["UnitPrice"] > 0)]
df = df.drop_duplicates()

df.info()

In [ ]:
nominal = df[["InvoiceDate", "StockCode", "Description","CustomerID", "Country"]]
ordinal = df[["InvoiceDate"]]
discrete = df[["Quantity"]]
continuous = df[["UnitPrice"]]

# Data visualization

### Customers per country

In [ ]:
cust_per_country = (
    df.groupby("Country")["CustomerID"]
      .nunique()
      .sort_values(ascending=False)
)

plt.figure(figsize=(10, 5))
sns.barplot(x=cust_per_country.index, y=cust_per_country.values,
            hue=cust_per_country.index, legend=False)
plt.title("Number of Unique Customers per Country")
plt.ylabel("Unique customers")
plt.xlabel("Country")
plt.xticks(rotation=90)
plt.tight_layout()
plt.yscale("log")
plt.show()

### Sales per countery

In [ ]:
df["Total"] = df["Quantity"] * df["UnitPrice"]

data = (
    df.groupby("Country")["Total"]
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)

plt.figure(figsize=(10, 5))
sns.barplot(data=data, x="Country", y="Total", hue=data.index, legend=False)
plt.xticks(rotation=90)
plt.title("Total Sales per Country")
plt.yscale("log")
plt.show()

# RFM

In [ ]:
CRFM = pd.DataFrame(df["CustomerID"].unique(), columns=["CustomerID"])

### Monetary

In [ ]:
CRFM = df.groupby("CustomerID")["Total"].sum().rename("Monetary").reset_index()
CRFM

### Recency

In [ ]:
last_sale = df.groupby("CustomerID")["InvoiceDate"].max()
CRFM["LastSale"] = CRFM["CustomerID"].map(last_sale)

snapshot = df["InvoiceDate"].max() + pd.Timedelta(days=1)
CRFM["Recency"] = (snapshot - CRFM["LastSale"]).dt.days
CRFM.drop(columns=["LastSale"], inplace=True)
CRFM

### Frequency

In [ ]:
frequency = df.groupby("CustomerID")["InvoiceNo"].nunique()
CRFM["Frequency"] = CRFM["CustomerID"].map(frequency)
CRFM

# RFM Visualization

In [ ]:
fig, axes = plt.subplots(nrows=1, ncols=3, figsize=(15, 4))

sns.histplot(CRFM["Recency"], ax=axes[0])
axes[0].set_title("Recency Distribution")

sns.histplot(CRFM["Frequency"], ax=axes[1], log_scale=True)
axes[1].set_title("Frequency Distribution (Log)")

sns.histplot(CRFM["Monetary"], ax=axes[2], log_scale=True)
axes[2].set_title("Monetary Distribution (Log)")

plt.tight_layout()
plt.show()

# Normalisation

In [ ]:
# RFM = (CRFM[["Recency", "Frequency", "Monetary"]] - CRFM[["Recency", "Frequency", "Monetary"]].min()) / (CRFM[["Recency", "Frequency", "Monetary"]].max() - CRFM[["Recency", "Frequency", "Monetary"]].min())
# CRFM

# Standariwation

In [ ]:
RFM = CRFM[["Monetary", "Frequency", "Recency"]].copy()

RFM["Monetary"] = np.log1p(RFM["Monetary"])
RFM["Frequency"] = np.log1p(RFM["Frequency"])
RFM["Recency"] = np.log1p(RFM["Recency"])

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

RFM = scaler.fit_transform(
    RFM
)
RFM = pd.DataFrame(
    RFM,
    columns=["Monetary", "Frequency", "Recency"]
)

# K-means

## Elbow method

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, pairwise_distances_argmin_min
from scipy.spatial.distance import cdist
import numpy as np

In [ ]:
inertias = []
mapping = {}
K = range(1, 20)

for k in K:
    kmeanModel = KMeans(n_clusters=k, random_state=42).fit(RFM)

    inertias.append(kmeanModel.inertia_)

    mapping[k] = inertias[-1]

In [ ]:
print("Inertia values:")
for key, val in mapping.items():
    print(f'{key} : {val}')

plt.plot(K, inertias, "bx-",color="red")
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Inertia')
plt.title('The Elbow Method using Inertia')
plt.grid()
plt.show()

### Elbow curve analysis
From the above graph, we can deduce that the best k-means sits between 3 <= k >= 7

## Silouhette score

In [ ]:
valid_K = range(2,11)
scors = {}
for k in valid_K:
    kmeanModel = KMeans(n_clusters=k, random_state=42, n_init=10)
    predicted_labels = kmeanModel.fit_predict(RFM)
    scors[k] = silhouette_score(RFM, predicted_labels)

In [ ]:
for s in scors:
    print(f"Silhouette score for {s} clusters: {scors[s]}")

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(list(scors.keys()), list(scors.values()), "rx-", c="y")
plt.xlabel("Number of Clusters (k)")
plt.ylabel("Silhouette score")
plt.title("Silhouette score graph")
plt.xticks(list(scors.keys()))
plt.grid()
plt.show()

### Silouhette score analysis
We guessed before that 3 <= k >= 7, but for testing pourpuses, we calculeted the score of (2,11).
The results was close enough to our previous guess, the best score belonged to k = 2 with a score of 0.722

### Difference between the best k (2) and the demanded one (3)
Although 2 was shown to be the best k, 3 would allow an Ecomerce buiseness which our data is from to handdle a more diverse pool of clients, as categorizing people to 3 groups with a right middle left mentality gives more options to make strategies to capture them than a plain either 1 or 2

In [ ]:
from sklearn.decomposition import PCA
pca = PCA(n_components=2)
rfmPCA = pca.fit_transform(RFM)
KModel = KMeans(n_clusters=3, random_state=42, n_init=10)
labels = KModel.fit_predict(RFM)

CRFM["Cluster"] = labels
rfmPCA

In [ ]:
# from sklearn.metrics.pairwise import pairwise_distances_argmin_min
# closest, test = pairwise_distances_argmin_min(KModel.cluster_centers_, rfm)
# test

# Cluster visualization

In [ ]:
u_labels = np.unique(labels)
for i in u_labels:
    plt.scatter(rfmPCA[labels == i , 0] , rfmPCA[labels == i , 1] , label = f"Cluster {i}", alpha=0.5, s=16)
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.legend()
plt.grid()
plt.show()

# Cluster centroids

In [ ]:
centroid = KModel.cluster_centers_
centroid = scaler.inverse_transform(centroid)
centroid = np.expm1(centroid)
centroid

In [ ]:
plt.scatter(
    rfmPCA[:,0],
    rfmPCA[:, 1],
    c=CRFM["Cluster"],
    alpha=0.5
)

plt.grid(True)
plt.show()

## RFM Cluster Profiling

In [ ]:
stats = CRFM.groupby("Cluster").agg(
    number_of_customers=("Cluster", "size"),
    avg_recency=("Recency", "mean"),
    median_recency=("Recency", "median"),
    avg_frequency=("Frequency", "mean"),
    median_frequency=("Frequency", "median"),
    avg_monetary=("Monetary", "mean"),
    median_monetary=("Monetary", "median"),
    total_revenue=("Monetary", "sum"),
).round(2)
stats.columns = pd.MultiIndex.from_tuples([
    ("Customers", "count"),
    ("Recency", "mean"), ("Recency", "median"),
    ("Frequency", "mean"), ("Frequency", "median"),
    ("Monetary", "mean"), ("Monetary", "median"), ("Monetary", "sum"),
])
stats

In [ ]:
stats.insert(0, "cluster_name", ["Loyal", "Lost", "Casual"])

In [ ]:
stats = stats.reindex([0,2,1])
stats

## Clusters observation
From the statistics above, we can easelly see that the 3 clusters are split into:

**Loyal:** Most cutomers, with the lowest recency score, and the highest frequency and monetary gains.

**Lost:** Customers who didn't buy in a long time, who also have the lowest frequency and monetary gains.

**Casual:** these are customers who sit in between, who have practically 2x better scores than the Lost group, with a frequency and monetary gain sitting right in the middle.

In [ ]:

import matplotlib.pyplot as plt
import numpy as np
from sklearn.cluster import DBSCAN
from sklearn.neighbors import NearestNeighbors

min_samples = 6
dist, test = NearestNeighbors(n_neighbors=min_samples).fit(RFM).kneighbors(RFM)
plt.plot(np.sort(dist[:, -1]))
plt.xlabel("Points (sorted)"); plt.ylabel(f"Distance to {min_samples}th neighbor")
plt.grid(True)
plt.show()
print(test)

db = DBSCAN(eps=0.4, min_samples=min_samples).fit(RFM)
labels = db.labels_

coords = PCA(n_components=2).fit_transform(RFM)
for lab in sorted(set(labels)):
    m = labels == lab
    if lab == -1:
        plt.scatter(coords[m, 0], coords[m, 1], c="k", marker="x", s=15, label="Noise")
    else:
        plt.scatter(coords[m, 0], coords[m, 1], s=15, alpha=0.6, label=f"Cluster {lab}")
plt.title(f"DBSCAN: {n_clusters} clusters")
plt.legend()
plt.grid(True)
plt.show()

mask = labels != -1
if n_clusters >= 2:
    print("Silhouette (no noise):", silhouette_score(RFM[mask], labels[mask]))

In [ ]:
from sklearn.metrics import calinski_harabasz_score

ch_scores = {}
for k in range(2, 11):
    labels = KMeans(n_clusters=k, random_state=42, n_init=10).fit_predict(RFM)
    ch_scores[k] = calinski_harabasz_score(RFM, labels)

plt.plot(list(ch_scores.keys()), list(ch_scores.values()), "go-")
plt.xlabel("Number of Clusters (k)")
plt.ylabel("Calinski-Harabasz score")
plt.grid()
plt.show()

In [ ]:
from sklearn.metrics import davies_bouldin_score

db_scores = {}
for k in range(2, 11):
    labels = KMeans(n_clusters=k, random_state=42, n_init=10).fit_predict(RFM)
    db_scores[k] = davies_bouldin_score(RFM, labels)

plt.plot(list(db_scores.keys()), list(db_scores.values()), "mo-")
plt.xlabel("Number of Clusters (k)")
plt.ylabel("Davies-Bouldin score")
plt.grid()
plt.show()